# Traning masif_ppi_search NN

### Prerequisites
- Training dataset of protein-protein complexes (PDB files)
- List files defining training/validation/test splits

### Step 1.1: Data Preparation

Prepare raw data (surfaces, features):
```bash
sbatch data_prepare.slurm
```

**What this does:**
- Downloads PDB files
- Triangulates molecular surfaces
- Computes geometric and chemical features
- Generates precomputation data (polar coordinates, etc.)

In [ ]:
# Run data_prepare.slurm
!sbatch data_prepare.slurm

In [ ]:
# TODO: Visualize training data

# 1. Read training list and testing list
with open("lists/training.txt", "r") as f:
    training_list = f.readlines()
training_list = [line.strip() for line in training_list]

with open("lists/testing.txt", "r") as f:
    testing_list = f.readlines()
testing_list = [line.strip() for line in testing_list]


# Function to visualize a model_id with py3Dmol
import py3Dmol
def visualize_model_id(model_id):
    pdb_id = model_id.split("_")[0]
    chain1 = model_id.split("_")[1]
    chain2 = model_id.split("_")[2]
    
    chain_1_pdb_file = f"data_preparation/01-benchmark_pdbs/{pdb_id}_{chain1}.pdb"
    chain_2_pdb_file = f"data_preparation/01-benchmark_pdbs/{pdb_id}_{chain2}.pdb"

    chain_1_pdb_str = open(chain_1_pdb_file, "r").read()
    chain_2_pdb_str = open(chain_2_pdb_file, "r").read()
    
    # View the structure
    view = py3Dmol.view(width=600, height=500)
    view.addModel(chain_1_pdb_str, "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "cyan"}})
    view.addModel(chain_2_pdb_str, "pdb")
    view.setStyle({"model": 1}, {"cartoon": {"color": "green"}})
    view.zoomTo()
    view.show()
    
# Example: visualize the first training model
model_idx = 0
model_id = training_list[model_idx]
visualize_model_id(model_id)

In [ ]:
model_idx += 1
model_id = training_list[model_idx]
print(model_id)
visualize_model_id(model_id)

### Step 1.2: Cache Training Data

Cache the training data for masif_ppi_search:
```bash
sbatch cache_nn.slurm  # Or run cache_nn.sh
```

**What this does:**
- Loads precomputed features for all training proteins
- Samples positive (binder) and negative (non-binder) pairs
- Saves cached data as `.npy` files

**Outputs:**
- `nn_models/sc05/cache/binder_*.npy` - Binder features
- `nn_models/sc05/cache/pos_*.npy` - Positive site features
- `nn_models/sc05/cache/neg_*.npy` - Negative site features
- `nn_models/sc05/cache/*_idx.npy` - Train/val/test indices

**Use the settings in `custom_params_mixed_5to1.py`**

Implemented the negative-sampling upgrade for `masif_ppi_search` with configurable mixed negatives and loss balancing.

Should sample more negatives than positives. 

In [ ]:
!sbatch cache_nn.slurm

In [ ]:
# Summary of cached features
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")  # or Path("masif/data/masif_ppi_search") if running from repo root
CACHE_DIR = DATA_DIR / "nn_models/sc05/cache"

def build_cache_summary():
    pos_names = np.load(CACHE_DIR / "pos_names.npy", allow_pickle=True)
    neg_names = np.load(CACHE_DIR / "neg_names.npy", allow_pickle=True)
    pos_train = set(np.load(CACHE_DIR / "pos_training_idx.npy"))
    pos_val = set(np.load(CACHE_DIR / "pos_val_idx.npy"))
    pos_test = set(np.load(CACHE_DIR / "pos_test_idx.npy"))
    neg_train = set(np.load(CACHE_DIR / "neg_training_idx.npy"))
    neg_val = set(np.load(CACHE_DIR / "neg_val_idx.npy"))
    neg_test = set(np.load(CACHE_DIR / "neg_test_idx.npy"))

    def split_for(i, train, val, test):
        if i in train: return "train"
        if i in val: return "val"
        if i in test: return "test"
        return "?"

    rows = []
    for i, name in enumerate(pos_names):
        rows.append({"type": "pos", "index": i, "split": split_for(i, pos_train, pos_val, pos_test), "name": name})
    for i, name in enumerate(neg_names):
        rows.append({"type": "neg", "index": i, "split": split_for(i, neg_train, neg_val, neg_test), "name": name})

    df = pd.DataFrame(rows)
    summary = df.groupby(["type", "split"]).size().unstack(fill_value=0)
    print("Counts by type and split:")
    display(summary)
    return df

In [ ]:
df_cache = build_cache_summary()
df_cache.head()


In [ ]:
# Cell: Visualize selected cached feature
import numpy as np
import py3Dmol
import pymesh
from pathlib import Path
from Bio.PDB import PDBParser

# --- Config: select entry to visualize ---
ENTRY_TYPE = "pos"   # "pos" or "neg"
ENTRY_INDEX = 0      # index into pos or neg array
SHOW_FULL_PATCH = True  # True = patch vertices, False = center only

# --- Paths (relative to masif_ppi_search data dir) ---
DATA_DIR = Path(".")
CACHE_DIR = DATA_DIR / "nn_models/sc05/cache"
PRECOMP_DIR = DATA_DIR / "data_preparation/04b-precomputation_12A/precomputation"
PLY_DIR = DATA_DIR / "data_preparation/01-benchmark_surfaces"
PDB_DIR = DATA_DIR / "data_preparation/01-benchmark_pdbs"

def parse_name(name):
    """Parse '1a2k_PB_PE_p1_123' -> (ppi_pair_id, pid, vix)"""
    parts = str(name).rsplit("_", 1)
    if len(parts) != 2:
        return None, None, None
    prefix, vix_str = parts
    try:
        vix = int(vix_str)
    except ValueError:
        return None, None, None
    sub = prefix.split("_")
    if len(sub) < 4:
        return None, None, None
    ppi_pair_id = "_".join(sub[:-2])
    pid = sub[-1]  # p1 or p2
    return ppi_pair_id, pid, vix

def get_patch_coords(ppi_pair_id, pid, vix, full_patch=False):
    """Get patch center (and optionally full patch) coordinates in original frame."""
    fields = ppi_pair_id.split("_")
    if len(fields) < 3:
        return None, None
    pdb_id, ch1, ch2 = fields[0], fields[1], fields[2]
    chain = ch1 if pid == "p1" else ch2
    ply_fn = PLY_DIR / f"{pdb_id}_{chain}.ply"
    if not ply_fn.exists():
        return None, None
    mesh = pymesh.load_mesh(str(ply_fn))
    center = mesh.vertices[vix]
    if not full_patch:
        return center, None
    list_idx_fn = PRECOMP_DIR / ppi_pair_id / f"{pid}_list_indices.npy"
    if not list_idx_fn.exists():
        return center, None
    neigh = np.load(list_idx_fn, allow_pickle=True)[vix]
    patch_coords = mesh.vertices[neigh]
    return center, patch_coords

def add_struct_to_py3dmol(structure, view=None):
    from Bio.PDB import PDBIO
    from io import StringIO
    io = PDBIO()
    io.set_structure(structure)
    buf = StringIO()
    io.save(buf)
    if view is None:
        view = py3Dmol.view(width=600, height=500)
    view.addModel(buf.getvalue(), "pdb")
    return view

def add_spheres(view, coords, color, radius=0.3):
    for pt in np.atleast_2d(coords):
        view.addSphere({"center": {"x": float(pt[0]), "y": float(pt[1]), "z": float(pt[2])}, "radius": radius, "color": color})

# --- Load and parse ---
names = np.load(CACHE_DIR / f"{ENTRY_TYPE}_names.npy", allow_pickle=True)
if ENTRY_INDEX >= len(names):
    print(f"Index {ENTRY_INDEX} out of range (max {len(names)-1})")
else:
    name = names[ENTRY_INDEX]
    ppi_pair_id, pid, vix = parse_name(name)
    if ppi_pair_id is None:
        print(f"Could not parse name: {name}")
    else:
        center, patch_coords = get_patch_coords(ppi_pair_id, pid, vix, full_patch=SHOW_FULL_PATCH)
        if center is None:
            print(f"Could not load coordinates for {name}")
        else:
            fields = ppi_pair_id.split("_")
            pdb_id, ch1, ch2 = fields[0], fields[1], fields[2]
            pdb_fn = PDB_DIR / f"{pdb_id}.pdb"  # or per-chain PDBs if you have them
            if pdb_fn.exists():
                parser = PDBParser(QUIET=True)
                struct = parser.get_structure("mol", str(pdb_fn))
                view = py3Dmol.view(width=600, height=500)
                add_struct_to_py3dmol(struct, view)
                add_spheres(view, center, "red", radius=0.5)
                if patch_coords is not None:
                    add_spheres(view, patch_coords, "blue", radius=0.2)
                view.zoomTo()
                view.show()
            print(f"{ENTRY_TYPE} #{ENTRY_INDEX}: {name} | split=... | center={center}")

### Step 1.3: Train MaSIF-ppi-search NN

Train the descriptor generation network:
```bash
sbatch masif_ppi_search_train.slurm
```

**Training configuration:**
- Network: Geometric CNN with rotation equivariance
- Loss: Siamese contrastive loss (push/pull descriptors)
- Output: 80-dimensional descriptors per vertex
- Duration: ~24-48 hours (depends on dataset size)

**Outputs:**
- `nn_models/sc05/all_feat/model_data/model.*` - Trained weights

In [ ]:
!sbatch masif_ppi_search_train.slurm

### Step 1.4: Generate Descriptors with New Model

Run inference with the newly trained model:
```bash
sbatch compute_descriptors.slurm
```

**What this does:**
- Loads the trained MaSIF-ppi-search weights
- Runs inference on all proteins
- Generates 80D descriptors for each surface vertex

**Expected output per protein complex:**
```
descriptors/sc05/all_feat/PDBID_CHAIN1_CHAIN2/
├── p1_desc_straight.npy    # Chain 1 descriptors (normal)
├── p1_desc_flipped.npy      # Chain 1 descriptors (flipped for complementarity)
├── p2_desc_straight.npy     # Chain 2 descriptors (normal)
└── p2_desc_flipped.npy      # Chain 2 descriptors (flipped)
```

In [ ]:
!sbatch compute_descriptors.slurm

In [ ]:
# Check the descriptors
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")  # or Path("masif/data/masif_ppi_search") if running from repo root
desc_dir = DATA_DIR / "descriptors/sc05/all_feat"


def load_descriptors(model_id):
    desc_fn = desc_dir / f"{model_id}/p1_desc_straight.npy"
    return np.load(desc_fn)

# Example: load descriptors for a specific model
model_id = "1CQI_A_B"
descriptors = load_descriptors(model_id)
print(descriptors.shape)  # Should print (n_vertices, 80)
